
# 3 &mdash; The verified reference table (App. F)

| target | minimal setup | result at $\Omega = 0$ |
|---|---|---|
| single-mode squeezing | 1 mode, on-site squeezing, 1 port | 12 dB (variance 0.062) |
| EPR entanglement | 2 modes, 1 squeezing edge, 2 ports | joint 0.062; local 8.0 |
| reciprocal squeezing | 2 modes, BS + squeezing edge, 2 ports | 5 dB at both ports (0.315) |
| directional squeezing | 2 signal + 1 damped aux | 7 dB / vacuum, fwd 1.30 |

Each row states the *setup* explicitly, so this notebook checks the forward map
and the oracle rather than the search: it pins the target, hands the oracle the
named graph, and prints what came out.

Script version: `examples/03_reference_table.py`.

In [1]:
# In case you are using Google Colab, this cell installs AutoGaussian
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install "git+https://github.com/Fouriersaur/AutoGaussian.git#subdirectory=autogaussian"
    !pip install cvxpy    # optional: infeasibility certificates (Sec. 6)
else:
    # running from a clone: put the package root on sys.path
    import os
    sys.path.insert(0, os.path.abspath(os.getcwd()))

In [2]:
import numpy as np

from autogaussian import (
    CovarianceArchitectureOptimizer,
    CovarianceTarget,
    TransmissionConstraint,
    duan_sum,
    variance_to_dB,
)

ROWS = []


def solve(target, graph_elements, num_auxiliary_modes=0, constraints=(), seed=0,
          num_tests=30, **kwargs):
    """Build the named graph, run the oracle on it, return (optimizer, best, success)."""
    optimizer = CovarianceArchitectureOptimizer(
        target, num_auxiliary_modes=num_auxiliary_modes, optimize_gauge=False,
        constraints=constraints, make_initial_test=False, seed=seed,
        kwargs_optimization={"num_tests": num_tests}, **kwargs)
    if graph_elements is None:
        graph = optimizer.space.fully_connected()
    else:
        graph = optimizer.space.empty()
        for element in graph_elements:
            graph[optimizer.space.slots.index(element[:3])] = element[3]
    success, infos = optimizer.test_graph(graph)
    return optimizer, infos[-1], success


## Row 1 &mdash; single-mode squeezing: 1 mode, on-site squeezing, 1 port

In [3]:
variance = 0.062
target = CovarianceTarget(num_ports=1)
target.pin((1, 1), variance)
target.pin((0, 0), 1.0 / variance)
target.pin((0, 1), 0.0)

optimizer, best, success = solve(target, [("onsite_squeezing", 0, 0, 2)])
V = np.real(optimizer.oracle.covariance(best["x"], 0.0))
print("   valid=%s   variance = %.4f  ->  %.2f dB   (target %.2f dB)"
      % (success, V[1, 1], variance_to_dB(V[1, 1]), variance_to_dB(variance)))
ROWS.append(("single-mode squeezing", success,
             "%.3f (%.1f dB)" % (V[1, 1], variance_to_dB(V[1, 1]))))

   valid=True   variance = 0.0620  ->  12.08 dB   (target 12.08 dB)



## Row 2 &mdash; EPR entanglement: 2 modes, 1 squeezing edge, 2 ports

In [4]:
local = 8.0
correlation = float(np.sqrt(local ** 2 - 1.0))     # pure two-mode squeezed vacuum
target = CovarianceTarget(num_ports=2)
target.pin_matrix([[local, 0.0, correlation, 0.0],
                   [0.0, local, 0.0, -correlation],
                   [correlation, 0.0, local, 0.0],
                   [0.0, -correlation, 0.0, local]])

optimizer, best, success = solve(target, [("two_mode_squeezing", 0, 1, 2)])
V = np.real(optimizer.oracle.covariance(best["x"], 0.0))
joint = 0.5 * (V[0, 0] + V[2, 2]) - V[0, 2]        # Var((x_1 - x_2)/sqrt(2))
print("   valid=%s   local = %.3f   joint quadrature = %.4f   Duan sum = %.4f (< 4)"
      % (success, V[0, 0], joint, duan_sum(V, 0, 1)))
ROWS.append(("EPR entanglement", success, "joint %.4f, local %.2f" % (joint, V[0, 0])))

   valid=True   local = 8.000   joint quadrature = 0.0627   Duan sum = 0.2510 (< 4)



## Row 3 &mdash; reciprocal squeezing: 2 modes, BS + squeezing edge, 2 ports

In [5]:
variance = 0.315
target = CovarianceTarget(num_ports=2)
target.pin((1, 1), variance)                       # p_1
target.pin((3, 3), variance)                       # p_2

optimizer, best, success = solve(
    target, [("beamsplitter", 0, 1, 2), ("two_mode_squeezing", 0, 1, 2)])
V = np.real(optimizer.oracle.covariance(best["x"], 0.0))
print("   valid=%s   port 1 = %.4f (%.2f dB)   port 2 = %.4f (%.2f dB)"
      % (success, V[1, 1], variance_to_dB(V[1, 1]), V[3, 3], variance_to_dB(V[3, 3])))
ROWS.append(("reciprocal squeezing", success, "%.3f / %.3f (%.1f dB both)"
             % (V[1, 1], V[3, 3], variance_to_dB(V[1, 1]))))

   valid=True   port 1 = 0.3150 (5.02 dB)   port 2 = 0.3150 (5.02 dB)



## Row 4 &mdash; directional squeezing: 2 signal modes + 1 damped auxiliary

This is the row App. F leaves open (*"on-site sqz + BS + phases"*), so the oracle
gets the fully connected 3-mode graph and we read off afterwards which elements
the solution actually used.

In [6]:
variance = 0.2                                     # ~7 dB
target = CovarianceTarget(num_ports=2)
target.pin_matrix([[1.0, 0.0, None, None],
                   [0.0, 1.0, None, None],
                   [None, None, variance, 0.0],
                   [None, None, 0.0, 1.0 / variance]])

optimizer, best, success = solve(
    target, None, num_auxiliary_modes=1,
    constraints=(TransmissionConstraint(1, 0, 1.3),), seed=3, num_tests=40)

minimal = optimizer.greedy_minimal_subgraph(optimizer.space.fully_connected(),
                                            num_tests=10)
print("   a minimal setup: %s" % ", ".join(optimizer.space.describe(minimal)))

best = optimizer.solution_of(minimal) or best
V = np.real(optimizer.oracle.covariance(best["x"], 0.0))
S, _ = optimizer.oracle.scattering(best["x"], 0.0)
print("   valid=%s   port 1 = [%.4f, %.4f] (vacuum)   port 2 = %.4f (%.2f dB)"
      % (success, V[0, 0], V[1, 1], V[2, 2], variance_to_dB(V[2, 2])))
print("   transport: forward %.3f   backward %.3f"
      % (abs(S[1, 0]) ** 2, abs(S[0, 1]) ** 2))
ROWS.append(("directional squeezing", success,
             "%.1f dB / vacuum, fwd %.2f, bwd %.2f"
             % (variance_to_dB(V[2, 2]), abs(S[1, 0]) ** 2, abs(S[0, 1]) ** 2)))

   a minimal setup: beam-splitter 0-2 (real), beam-splitter 1-2 (real), detuning Delta_2, on-site squeezing 0 (complex), two-mode squeezing 0-1 (complex), two-mode squeezing 0-2 (complex), on-site squeezing 1 (complex), two-mode squeezing 1-2 (complex), on-site squeezing 2 (complex)


   valid=True   port 1 = [1.0000, 1.0000] (vacuum)   port 2 = 0.2000 (6.99 dB)
   transport: forward 1.300   backward 0.774



## The table

In [7]:
print("%-24s %-8s %s" % ("target", "valid", "result at Omega = 0"))
print("-" * 72)
for name, ok, result in ROWS:
    print("%-24s %-8s %s" % (name, ok, result))
print("=" * 72)
print("all rows valid:", all(row[1] for row in ROWS))

target                   valid    result at Omega = 0
------------------------------------------------------------------------
single-mode squeezing    True     0.062 (12.1 dB)
EPR entanglement         True     joint 0.0627, local 8.00
reciprocal squeezing     True     0.315 / 0.315 (5.0 dB both)
directional squeezing    True     7.0 dB / vacuum, fwd 1.30, bwd 0.77
all rows valid: True
